# Phase 8c — Part A, controlled: switch rate vs *controlled* chain-of-thought length

Phase 8b measured think length passively and hit two walls (the answer was
token-starved, and language-switching barely reproduces on OpenRouter with
thinking on). This does the clean version: it **controls** the amount of
thinking across levels (off → low → medium → high), **measures** the think
length each level actually produces, and plots the **switch rate vs the
measured think length**.

The hypothesis, restated as a controlled experiment:

> if enough thinking *suppresses* the language switch, the switch rate falls as
> the think length grows — a critical length `L*`. The `off` level (thinking
> disabled) is the RESULTS.md regime, where the behaviour reproduces; the
> higher levels test whether more deliberation shuts the tip down.

The key analysis **bins by the measured think length** (pooled over all levels),
so it is robust even if a provider maps the effort levels coarsely — the levels
only need to produce a *spread* of think lengths, which off-vs-on guarantees.

**No GPU, no draft.** OpenRouter generation + text analysis. Needs an OpenRouter
key and an HF token (WeirdChat prompts + Qwen tokenizer), read from Colab secrets.


In [ ]:
# === Cell 1 — config =========================================================
import os
try:
    from google.colab import userdata
    for k in ("OPENROUTER_API_KEY","HF_TOKEN"):
        v=None
        try: v=userdata.get(k)
        except Exception: v=None
        if v: os.environ.setdefault(k,v)
except Exception as e:
    print("colab secrets unavailable:", e)

MODEL       = "qwen/qwen3.6-35b-a3b"
BEHAVIORS   = ["language-switching-english"]
PATTERNS    = 6
PROMPTS     = 8
SAMPLES     = 48      # completions per (prompt, level). Total = prompts x LEVELS x SAMPLES.
TEMPERATURE = 1.0
MAX_TOKENS  = 8000    # reasoning + answer share this; large so the answer is never starved
CONCURRENCY = 24

# The control knob. Each level is (name, extra_body). "off" disables thinking
# (the RESULTS.md regime); the others ask for progressively more reasoning.
# Effort is the most portable control; if a provider ignores it the levels just
# collapse to fewer distinct measured lengths — the analysis bins by the MEASURED
# length, so it still works. Token-budget alternative is commented below.
LEVELS = [
    ("off",    {"reasoning": {"enabled": False}}),
    ("low",    {"reasoning": {"effort": "low"}}),
    ("medium", {"reasoning": {"effort": "medium"}}),
    ("high",   {"reasoning": {"effort": "high"}}),
]
# LEVELS = [("off",{"reasoning":{"enabled":False}}),
#           ("t128",{"reasoning":{"max_tokens":128}}),
#           ("t512",{"reasoning":{"max_tokens":512}}),
#           ("t2048",{"reasoning":{"max_tokens":2048}})]   # token-budget variant

FOREIGN_RUN = 3
OUTPUT      = "/content/drive/MyDrive/weirdspec/reasoning_sweep.jsonl"   # resumable
MOUNT_DRIVE = True

assert os.environ.get("OPENROUTER_API_KEY"), "set OPENROUTER_API_KEY (Colab secret or env)"
print(f"plan: {PATTERNS}x{PROMPTS} prompts x {len(LEVELS)} levels x {SAMPLES} samples "
      f"= up to {PATTERNS*PROMPTS*len(LEVELS)*SAMPLES} completions, T={TEMPERATURE}")


In [ ]:
# === Cell 2 — mount Drive + install weirdchat (no GPU, no torch) ============
import os, sys, subprocess, importlib
if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e:
        print("drive mount skipped:", e)
if not os.path.exists("/content/WeirdChat"):
    subprocess.run(["git","clone","--depth","1","--branch",
                    "main",
                    "https://github.com/Erikiss/WeirdChat","/content/WeirdChat"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","/content/WeirdChat","nest_asyncio"],
               check=True)
if "/content/WeirdChat" not in sys.path:
    sys.path.insert(0, "/content/WeirdChat")
importlib.invalidate_caches()
import weirdchat as _wc
print("weirdchat OK:", os.path.dirname(_wc.__file__))


In [ ]:
# === Cell 3 — select the language-switching prompts =========================
import weirdchat as wc
sel=[]
for beh in BEHAVIORS:
    pats = wc.patterns(behavior_id=beh, subject_model=MODEL)
    pats.sort(key=lambda p:(p.openrouter_replication.rate if p.openrouter_replication else 0) or 0,
              reverse=True)
    for p in pats[:PATTERNS]:
        for pr in wc.prompts(p)[:PROMPTS]:
            sel.append((beh, p.pattern_id, pr))
print(f"{len(sel)} prompt cells; up to {len(sel)*len(LEVELS)*SAMPLES} completions")
if sel:
    print("top pattern reproduction rates:",
          [round((p.openrouter_replication.rate if p.openrouter_replication else 0) or 0,2)
           for p in pats[:PATTERNS]])


In [ ]:
# === Cell 4 — generate across levels (async, resumable) =====================
import json, os, re, hashlib, asyncio
from openai import AsyncOpenAI

def read_jsonl(path):
    rows=[]
    if os.path.exists(path):
        with open(path,encoding="utf-8") as f:
            for line in f:
                line=line.strip()
                if line: rows.append(json.loads(line))
    return rows
def prompt_key(pr):
    return hashlib.sha1("||".join(m.role+":"+m.content for m in pr.messages).encode()).hexdigest()[:12]
def get_reasoning(msg):
    r=getattr(msg,"reasoning",None); me=getattr(msg,"model_extra",None) or {}
    if r is None: r=me.get("reasoning")
    if r is None:
        det=me.get("reasoning_details") or getattr(msg,"reasoning_details",None)
        if det:
            try: r="".join(d.get("text","") for d in det if isinstance(d,dict))
            except Exception: r=None
    return r
def split_think(msg):
    content=msg.content or ""; r=get_reasoning(msg)
    if r: return r, content, "reasoning_field"
    m=re.search(r"<think>(.*?)</think>(.*)", content, re.DOTALL)
    if m: return m.group(1), m.group(2), "inline_think"
    return "", content, "no_think"

client=AsyncOpenAI(base_url="https://openrouter.ai/api/v1",
                   api_key=os.environ["OPENROUTER_API_KEY"], timeout=240, max_retries=3)
done={(r["pattern_id"],r["prompt_key"],r["level"],r["sample_idx"]) for r in read_jsonl(OUTPUT)}
os.makedirs(os.path.dirname(OUTPUT) or ".", exist_ok=True)
_outf=open(OUTPUT,"a",encoding="utf-8"); _lock=asyncio.Lock(); _sem=asyncio.Semaphore(CONCURRENCY)

async def one(beh,pid,pr,level,eb,k):
    pk=prompt_key(pr)
    if (pid,pk,level,k) in done: return
    async with _sem:
        try:
            comp=await client.chat.completions.create(
                model=MODEL, messages=[{"role":m.role,"content":m.content} for m in pr.messages],
                temperature=TEMPERATURE, max_tokens=MAX_TOKENS, extra_body=eb)
        except Exception as e:
            print(f"  gen error [{level}]:", type(e).__name__); return
    reasoning,answer,mode=split_think(comp.choices[0].message)
    rec=dict(behavior=beh,pattern_id=pid,prompt_key=pk,level=level,sample_idx=k,mode=mode,
             reasoning=reasoning[:8000],answer=answer[:8000],
             reasoning_chars=len(reasoning),answer_chars=len(answer))
    async with _lock:
        _outf.write(json.dumps(rec,ensure_ascii=False)+"\n"); _outf.flush()

async def run_all():
    tasks=[one(beh,pid,pr,level,eb,k)
           for (beh,pid,pr) in sel for (level,eb) in LEVELS for k in range(SAMPLES)]
    for i in range(0,len(tasks),200):
        await asyncio.gather(*tasks[i:i+200]); print(f"  {min(i+200,len(tasks))}/{len(tasks)}")

import nest_asyncio; nest_asyncio.apply()
asyncio.get_event_loop().run_until_complete(run_all())
_outf.close(); print("saved ->", OUTPUT)


In [ ]:
# === Cell 5 — measure think length + non-Latin onset ========================
import re
from collections import Counter
TOK=None
try:
    from transformers import AutoTokenizer
    TOK=AutoTokenizer.from_pretrained("Qwen/Qwen3.6-35B-A3B", trust_remote_code=True); UNIT="tokens"
except Exception as e:
    UNIT="words"; print("tokenizer unavailable, L_cot in words:", type(e).__name__)
def length_of(t):
    if not t: return 0
    return len(TOK(t)["input_ids"]) if TOK else len(t.split())
FOREIGN=[(0x0370,0x03FF),(0x0400,0x04FF),(0x0500,0x052F),(0x0530,0x058F),(0x0590,0x05FF),
         (0x0600,0x06FF),(0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),
         (0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_foreign(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    if o<0x0250: return False
    return any(a<=o<=b for a,b in FOREIGN)
def foreign_onset(text, run=FOREIGN_RUN):
    cnt=0
    for ch in text:
        if is_foreign(ch):
            cnt+=1
            if cnt>=run: return True
        elif ch.isalpha():
            cnt=0
    return False

rows=read_jsonl(OUTPUT)
for r in rows:
    r["L_cot"]=length_of(r["reasoning"])
    r["switched"]=foreign_onset(r["answer"])
import numpy as np
emp=np.mean([r["answer_chars"]<20 for r in rows]) if rows else 0.0
print(f"{len(rows)} completions | switched: {sum(r['switched'] for r in rows)} | "
      f"empty answers: {100*emp:.0f}% | modes: {dict(Counter(r['mode'] for r in rows))} | unit: {UNIT}")
print("per-level counts:", dict(Counter(r["level"] for r in rows)))


In [ ]:
# === Cell 6 — switch rate vs think length (the controlled test) =============
import numpy as np, matplotlib.pyplot as plt
def wilson(k,n,z=1.96):
    if n==0: return 0.0,0.0,0.0
    p=k/n; d=1+z*z/n
    c=(p+z*z/(2*n))/d; h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return p, max(0,c-h), min(1,c+h)
def diff_ci(k1,n1,k2,n2,z=1.96):
    p1,p2=k1/max(n1,1),k2/max(n2,1); se=np.sqrt(p1*(1-p1)/max(n1,1)+p2*(1-p2)/max(n2,1))
    return p1-p2, (p1-p2)-z*se, (p1-p2)+z*se

good=[r for r in rows if r["answer_chars"]>=20]     # answers that actually exist
n_sw=sum(r["switched"] for r in good)
print(f"usable (non-empty answers): {len(good)} / {len(rows)}   total switched: {n_sw}")

if n_sw < 15:
    print(f"\n!! only {n_sw} switches across the whole sweep — too few to fit a curve.")
    print("   Even the 'off' level barely reproduces language-switching on OpenRouter.")
    print("   This is the wall to self-host the FP8 subject (single-GPU SkyPilot YAML) and")
    print("   re-point MODEL/base-url at it; the behaviour reproduces properly there.")
else:
    # (A) per-level: does the control knob actually move the think length?
    order=[lv for lv,_ in LEVELS]
    print(f"\n{'level':8s} {'n':>5} {'switch%':>8} {'95% CI':>16} {'med think':>10} {'empty%':>7}")
    for lv in order:
        g=[r for r in good if r["level"]==lv]; allg=[r for r in rows if r["level"]==lv]
        k=sum(r["switched"] for r in g); p,lo,hi=wilson(k,len(g))
        ml=np.median([r["L_cot"] for r in g]) if g else 0
        emp=100*np.mean([r["answer_chars"]<20 for r in allg]) if allg else 0
        print(f"{lv:8s} {len(g):5d} {100*p:7.1f}% [{100*lo:5.1f},{100*hi:5.1f}] {ml:10.0f} {emp:6.0f}%")

    # (B) the robust analysis: bin ALL usable completions by MEASURED think length.
    L=np.array([r["L_cot"] for r in good],float); S=np.array([r["switched"] for r in good])
    edges=np.unique(np.quantile(L, np.linspace(0,1,7)))
    xs,ps,los,his,ns=[],[],[],[],[]
    for a,b in zip(edges[:-1],edges[1:]):
        m=(L>=a)&(L<=b) if b==edges[-1] else (L>=a)&(L<b)
        if m.sum()<10: continue
        k=int(S[m].sum()); p,lo,hi=wilson(k,int(m.sum()))
        xs.append(float(np.median(L[m]))); ps.append(p); los.append(lo); his.append(hi); ns.append(int(m.sum()))
    xs=np.array(xs); ps=np.array(ps)
    print(f"\nswitch rate binned by measured think length ({UNIT}):")
    for x,p,n in zip(xs,ps,ns): print(f"  think~{x:6.0f}: {100*p:5.1f}%  (n={n})")

    # verdict: shortest-think regime vs longest-think regime
    lowmask=L<=np.quantile(L,0.25); himask=L>=np.quantile(L,0.75)
    d,dlo,dhi=diff_ci(int(S[lowmask].sum()),int(lowmask.sum()),int(S[himask].sum()),int(himask.sum()))
    if dlo>0:   verdict=f"short-think switch rate is HIGHER by {100*d:.1f}pp [{100*dlo:.1f},{100*dhi:.1f}] => thinking SUPPRESSES the switch (critical length L*)"
    elif dhi<0: verdict=f"short-think rate is LOWER ({100*d:.1f}pp) => more thinking INCREASES switching (against L*)"
    else:       verdict=f"difference {100*d:.1f}pp, 95% CI [{100*dlo:.1f},{100*dhi:.1f}] straddles 0 => inconclusive"
    print("\nVERDICT:", verdict)

    fig,ax=plt.subplots(1,2,figsize=(11,4))
    yerr=np.vstack([ps-los,his-ps])
    ax[0].errorbar(xs,100*ps,yerr=100*yerr,fmt="o-",capsize=3)
    ax[0].set_xlabel(f"measured think length ({UNIT})"); ax[0].set_ylabel("switch rate (%)")
    ax[0].set_title("switch rate vs think length"); ax[0].set_ylim(bottom=0)
    lv_x=[np.median([r["L_cot"] for r in good if r["level"]==lv]) for lv in order]
    lv_p=[wilson(sum(r["switched"] for r in good if r["level"]==lv),
                 len([r for r in good if r["level"]==lv]))[0] for lv in order]
    ax[1].bar(range(len(order)),[100*p for p in lv_p]); ax[1].set_xticks(range(len(order))); ax[1].set_xticklabels(order)
    ax[1].set_ylabel("switch rate (%)"); ax[1].set_title("switch rate by control level")
    plt.tight_layout(); plt.show()


### How to read it

* **The curve (left)** is the whole point: switch rate vs the *measured* think
  length. A downward slope — short thinking switches often, long thinking rarely —
  is the critical-length signature: enough deliberation shuts the tip down. The
  **VERDICT** line makes it explicit by comparing the shortest-think quartile to
  the longest-think quartile with a 95% CI on the difference.
* **The per-level table** shows whether the control knob (`off`/`low`/`medium`/
  `high`) actually moved the think length. If `low/medium/high` collapse to the
  same length, the provider ignores effort — but the binned curve still holds,
  because it bins by measured length, and `off` still anchors the short end.
* **Empty-answer %** should be near 0 now (MAX_TOKENS=8000). If it climbs at the
  `high` level, raise MAX_TOKENS further.
* If **total switches < 15**, even the `off` level doesn't reproduce enough on
  OpenRouter — that is the point to self-host the FP8 subject and re-point `MODEL`.

This is the controlled version of Part A: it turns "does short CoT cause the
switch?" into a switch-rate-vs-length curve you can actually read.
